In [ ]:
%load_ext autoreload
%autoreload 2

%aimport -faiss
%aimport -PIL

import sys
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from hep_tracking.ann_models import FaissIVFFlat, FaissIVFPQ, HnswGraph
from hep_tracking.utils import calculate_recall

print("Zależności ANN załadowane pomyślnie!")

In [ ]:
data_dir = project_root / "data"
dataset_name = "dataset_hard_100k.npz"
candidates_name = "candidates_hard_100k.npz"

data_path = data_dir / dataset_name
candidates_path = data_dir / candidates_name

if not data_path.exists() or not candidates_path.exists():
    raise FileNotFoundError("Brak danych! Upewnij się, że wywołałeś `make candidates`.")

# Wczytujemy cechy hitów
features = np.load(data_path)["X"]
n_queries = features.shape[0]

# Wczytujemy idealne indeksy kNN jako nasz punkt odniesienia
true_indices = np.load(candidates_path)["indices"]
k_neighbors = true_indices.shape[1]

print(f"Załadowano dane: {n_queries} punktów w przestrzeni {features.shape[1]}D")
print(f"Szukamy {k_neighbors} najbliższych sąsiadów.")

In [ ]:
def evaluate_ann_model(model_name, model_instance, features, true_indices, k):
    """Buduje indeks, wykonuje zapytanie i zwraca metryki (QPS, Recall)."""
    print(f"Trenowanie i budowa indeksu: {model_name}...")
    
    model_instance.build(features)
    
    start_time = time.perf_counter()
    _, pred_indices = model_instance.query(features, k)
    query_time = time.perf_counter() - start_time
    
    qps = features.shape[0] / query_time
    recall = calculate_recall(true_indices, pred_indices)
    
    print(f" -> QPS: {qps:,.0f} | Recall: {recall:.4f}\n")
    return qps, recall

In [ ]:
#Okazuje się, że FAISS chce mieć liczbę wymiarów będącą 2^n, a my mamy 5, więc albo musimy zrobić PCA do 4 (albo jakoś inaczej) albo dodać padding do 8. Na razie trzeba na CPU
USE_GPU = False

results = {
    "IVFFlat": {"recall": [], "qps": [], "labels": []},
    "IVFPQ": {"recall": [], "qps": [], "labels": []},
    "HNSW": {"recall": [], "qps": [], "labels": []}
}

nlist = 100
for nprobe in [1, 2, 5, 10, 20, 50]:
    model = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=USE_GPU)
    qps, recall = evaluate_ann_model(f"IVFFlat (nprobe={nprobe})", model, features, true_indices, k_neighbors)
    results["IVFFlat"]["qps"].append(qps)
    results["IVFFlat"]["recall"].append(recall)
    results["IVFFlat"]["labels"].append(f"np={nprobe}")

for nprobe in [1, 2, 5, 10, 20, 50, 100]:
    model = FaissIVFPQ(nlist=nlist, m=5, nprobe=nprobe, use_gpu=USE_GPU)
    qps, recall = evaluate_ann_model(f"IVFPQ (nprobe={nprobe})", model, features, true_indices, k_neighbors)
    results["IVFPQ"]["qps"].append(qps)
    results["IVFPQ"]["recall"].append(recall)
    results["IVFPQ"]["labels"].append(f"np={nprobe}")

m_graph = 16
for ef in [10, 20, 50, 100, 200]:
    model = HnswGraph(m=m_graph, ef_construction=200, ef=ef)
    qps, recall = evaluate_ann_model(f"HNSW (ef={ef})", model, features, true_indices, k_neighbors)
    results["HNSW"]["qps"].append(qps)
    results["HNSW"]["recall"].append(recall)
    results["HNSW"]["labels"].append(f"ef={ef}")

In [ ]:
plt.figure(figsize=(10, 7))

colors = {"IVFFlat": "blue", "IVFPQ": "green", "HNSW": "red"}
markers = {"IVFFlat": "o", "IVFPQ": "s", "HNSW": "^"}

for name, data in results.items():
    plt.plot(data["recall"], data["qps"], 
             marker=markers[name], color=colors[name], 
             linestyle='-', linewidth=2, markersize=8, label=name)
    
    for i, label in enumerate(data["labels"]):
        if i % 2 == 0 or i == len(data["labels"]) - 1:
            plt.annotate(label, (data["recall"][i], data["qps"][i]), 
                         textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, alpha=0.7)

plt.yscale("log")
plt.xlabel("Recall (Czułość względem exact kNN)", fontsize=12)
plt.ylabel("QPS - Queries Per Second (Wyżej = Szybciej)", fontsize=12)
plt.title("Wydajność algorytmów ANN: Recall vs. QPS (Przestrzeń 5D)", fontsize=14)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=11)
plt.xlim(0.8, 1.05)
plt.tight_layout()
plt.show()